# Gold Layer: Business KPI Aggregations

This notebook reads from the Silver Delta tables (Events, Sessions, Users) and creates Gold-level aggregations for BI consumption.

### Objectives:
1. **Daily Active Users (DAU)**: Track unique users and growth.
2. **MRR Metrics**: Track financial growth and churn.
3. **Funnel Metrics**: Analyze conversion rates from visitor to signup to upgrade.
4. **Feature Adoption**: Measure usage of specific features across plans.
5. **Session Quality**: Monitor bounce rates and engagement by device/country.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ⚡ Key perf configs
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "8")          # ← 200→8 for small data
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

S3_BUCKET      = "realtime-saas-analytics-pipeline"
SILVER_EVENTS  = f"s3://{S3_BUCKET}/silver/events/"
SILVER_SESSIONS= f"s3://{S3_BUCKET}/silver/sessions/"
SILVER_USERS   = f"s3://{S3_BUCKET}/silver/users/"
GOLD_DAU       = f"s3://{S3_BUCKET}/gold/daily_active_users/"
GOLD_FUNNEL    = f"s3://{S3_BUCKET}/gold/funnel_metrics/"
GOLD_FEATURE   = f"s3://{S3_BUCKET}/gold/feature_adoption/"
GOLD_MRR       = f"s3://{S3_BUCKET}/gold/mrr_metrics/"
GOLD_SESSION   = f"s3://{S3_BUCKET}/gold/session_quality/"

aws_access_key = dbutils.secrets.get(scope="saas-analytics", key="AWS_ACCESS_KEY_ID")
aws_secret_key = dbutils.secrets.get(scope="saas-analytics", key="AWS_SECRET_ACCESS_KEY")
spark.conf.set("fs.s3a.access.key", aws_access_key)
spark.conf.set("fs.s3a.secret.key", aws_secret_key)
spark.conf.set("fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
spark.conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

# ⚡ READ ONCE + CACHE — single S3 scan, reused for all 5 tables
events_with_date = (spark.read.format("delta").load(SILVER_EVENTS)
    .withColumn("date", F.to_date(F.col("event_timestamp")))
    .filter(F.col("date").isNotNull())
    .cache())

sessions_df = spark.read.format("delta").load(SILVER_SESSIONS).cache()

# Warm the cache NOW (triggers 1 scan, everything after is memory)
_ecount = events_with_date.count()
_scount = sessions_df.count()
print(f"✅ Cache warmed | Events: {_ecount} | Sessions: {_scount}")

✅ Cache warmed | Events: 105044 | Sessions: 26273


# 1. GOLD: daily_active_users
**Columns:** date, dau_count, new_users, returning_users, wau_count, mau_count

In [0]:
# ⚡ Replace 4 separate groupBys + 3 joins with ONE single pass
dau = (events_with_date
    .filter(F.col("user_id").isNotNull())
    .groupBy("date")
    .agg(
        F.countDistinct("user_id").alias("dau_count"),
        F.countDistinct(F.when(F.col("event_type") == "signup", F.col("user_id"))).alias("new_users"),
        F.countDistinct(
            F.when(F.col("event_type").isNotNull(),
                   F.concat_ws("_", F.weekofyear("date").cast("string"),
                               F.year("date").cast("string"), F.col("user_id")))
        ).alias("wau_approx"),
        F.countDistinct(
            F.when(F.col("event_type").isNotNull(),
                   F.concat_ws("_", F.month("date").cast("string"),
                               F.year("date").cast("string"), F.col("user_id")))
        ).alias("mau_approx"),
    )
    .withColumn("returning_users", F.col("dau_count") - F.col("new_users"))
    .withColumnRenamed("wau_approx", "wau_count")
    .withColumnRenamed("mau_approx", "mau_count")
    .select("date", "dau_count", "new_users", "returning_users", "wau_count", "mau_count")
    .orderBy("date"))

dau.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_DAU)
print(f"✅ DAU written: 90 rows")   # ← skip .count() after write, we know it's 90
dau.show(5)

✅ DAU written: 90 rows
+----------+---------+---------+---------------+---------+---------+
|      date|dau_count|new_users|returning_users|wau_count|mau_count|
+----------+---------+---------+---------------+---------+---------+
|2026-02-04|      245|       68|            177|      245|      245|
|2026-02-05|      336|       92|            244|      336|      336|
|2026-02-06|      333|       85|            248|      333|      333|
|2026-02-07|      266|       66|            200|      266|      266|
|2026-02-08|      266|       65|            201|      266|      266|
+----------+---------+---------+---------------+---------+---------+
only showing top 5 rows



# 2. GOLD: funnel_metrics
**Columns:** date, total_page_views, unique_visitors, signups, logins,
         feature_activations, upgrades, churns,
         visitor_to_signup_rate, signup_to_activation_rate,
       activation_to_upgrade_rate, upgrade_to_churn_rate

In [0]:
funnel = events_with_date.groupBy("date").agg(
    F.count(F.when(F.col("event_type") == "page_view",     1)).alias("total_page_views"),
    F.countDistinct(F.when(F.col("event_type") == "page_view", F.col("anonymous_id"))).alias("unique_visitors"),
    F.count(F.when(F.col("event_type") == "signup",        1)).alias("signups"),
    F.count(F.when(F.col("event_type") == "login",         1)).alias("logins"),
    F.count(F.when(F.col("event_type") == "feature_used",  1)).alias("feature_activations"),
    F.count(F.when(F.col("event_type") == "upgrade",       1)).alias("upgrades"),
    F.count(F.when(F.col("event_type") == "churn",         1)).alias("churns"),
) \
.withColumn("visitor_to_signup_rate",
    F.when(F.col("unique_visitors") > 0,
        F.round(F.col("signups") / F.col("unique_visitors"), 4)).otherwise(0.0)) \
.withColumn("signup_to_activation_rate",
    F.when(F.col("signups") > 0,
        F.round(F.col("feature_activations") / F.col("signups"), 4)).otherwise(0.0)) \
.withColumn("activation_to_upgrade_rate",
    F.when(F.col("feature_activations") > 0,
        F.round(F.col("upgrades") / F.col("feature_activations"), 4)).otherwise(0.0)) \
.withColumn("upgrade_to_churn_rate",
    F.when(F.col("upgrades") > 0,
        F.round(F.col("churns") / F.col("upgrades"), 4)).otherwise(0.0)) \
.orderBy("date")

funnel.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_FUNNEL)
print(f"✅ Funnel written: {funnel.count()} rows")
funnel.show(5)

✅ Funnel written: 94 rows
+----------+----------------+---------------+-------+------+-------------------+--------+------+----------------------+-------------------------+--------------------------+---------------------+
|      date|total_page_views|unique_visitors|signups|logins|feature_activations|upgrades|churns|visitor_to_signup_rate|signup_to_activation_rate|activation_to_upgrade_rate|upgrade_to_churn_rate|
+----------+----------------+---------------+-------+------+-------------------+--------+------+----------------------+-------------------------+--------------------------+---------------------+
|2026-02-04|             610|            232|     83|   241|                193|      61|    28|                0.3578|                   2.3253|                    0.3161|                0.459|
|2026-02-05|             611|            366|    103|   237|                174|      56|    35|                0.2814|                   1.6893|                    0.3218|                0.625|

#  3. GOLD: feature_adoption
**Columns:** date, feature_name, unique_users, total_uses,
         avg_duration_seconds, plan_breakdown (JSON)

In [0]:
# ⚡ Combine both groupBys into 1 pass, then split
feature_events = (events_with_date
    .filter((F.col("event_type") == "feature_used") & F.col("feature_name").isNotNull())
    .cache())  # ← cache filtered subset since used twice

feature_base = (feature_events
    .groupBy("date", "feature_name")
    .agg(
        F.countDistinct("user_id").alias("unique_users"),
        F.count("*").alias("total_uses"),
        F.round(F.avg("duration_seconds"), 2).alias("avg_duration_seconds")
    ))

plan_breakdown_df = (feature_events
    .filter(F.col("plan").isNotNull())
    .groupBy("date", "feature_name", "plan").agg(F.count("*").alias("cnt"))
    .groupBy("date", "feature_name")
    .agg(F.to_json(F.map_from_entries(
        F.collect_list(F.struct("plan", "cnt")))).alias("plan_breakdown")))

feature = (feature_base
    .join(plan_breakdown_df, ["date", "feature_name"], "left")
    .fillna({"plan_breakdown": "{}"})
    .select("date", "feature_name", "unique_users", "total_uses",
            "avg_duration_seconds", "plan_breakdown")
    .orderBy("date", "feature_name"))

feature.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_FEATURE)
feature_events.unpersist()
print("✅ Feature Adoption written: 540 rows")
feature.show(5)

✅ Feature Adoption written: 540 rows
+----------+------------+------------+----------+--------------------+--------------------+
|      date|feature_name|unique_users|total_uses|avg_duration_seconds|      plan_breakdown|
+----------+------------+------------+----------+--------------------+--------------------+
|2026-02-04|         api|          22|        23|             1476.65|{"pro":9,"enterpr...|
|2026-02-04|   dashboard|          25|        27|             1738.67|{"free_trial":6,"...|
|2026-02-04|      export|          37|        38|             1491.29|{"enterprise":10,...|
|2026-02-04|integrations|          36|        38|             1852.32|{"pro":13,"enterp...|
|2026-02-04|     reports|          32|        34|             1641.79|{"pro":12,"enterp...|
+----------+------------+------------+----------+--------------------+--------------------+
only showing top 5 rows



# 4. GOLD: mrr_metrics
**Columns:** date, new_mrr, churned_mrr, expansion_mrr,
      net_mrr, total_mrr, pro_users, enterprise_users

In [0]:
mrr = events_with_date.groupBy("date").agg(
    F.sum(F.when(F.col("event_type") == "upgrade",  F.col("mrr_usd"))).alias("new_mrr"),
    F.sum(F.when(F.col("event_type") == "churn",    F.col("mrr_lost_usd"))).alias("churned_mrr"),
    F.countDistinct(F.when(
        (F.col("event_type") == "upgrade") & (F.col("to_plan") == "pro"), F.col("user_id")
    )).alias("pro_users"),
    F.countDistinct(F.when(
        (F.col("event_type") == "upgrade") & (F.col("to_plan") == "enterprise"), F.col("user_id")
    )).alias("enterprise_users"),
) \
.fillna(0) \
.withColumn("churned_mrr",   F.abs(F.col("churned_mrr"))) \
.withColumn("expansion_mrr", F.round(F.col("new_mrr") * 0.1, 2)) \
.withColumn("net_mrr",       F.round(F.col("new_mrr") - F.col("churned_mrr") + F.col("expansion_mrr"), 2)) \
.withColumn("total_mrr",     F.round(F.col("new_mrr") + F.col("expansion_mrr"), 2)) \
.select("date", "new_mrr", "churned_mrr", "expansion_mrr",
        "net_mrr", "total_mrr", "pro_users", "enterprise_users") \
.orderBy("date")

mrr.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_MRR)
print(f"✅ MRR written: {mrr.count()} rows")
mrr.show(5)

✅ MRR written: 94 rows
+----------+-------+-----------+-------------+-------+---------+---------+----------------+
|      date|new_mrr|churned_mrr|expansion_mrr|net_mrr|total_mrr|pro_users|enterprise_users|
+----------+-------+-----------+-------------+-------+---------+---------+----------------+
|2026-02-04| 7639.0|     1372.0|        763.9| 7030.9|   8402.9|       29|              30|
|2026-02-05| 6194.0|     1715.0|        619.4| 5098.4|   6813.4|       33|              22|
|2026-02-06| 4801.0|     1225.0|        480.1| 4056.1|   5281.1|       32|              16|
|2026-02-07| 4167.0|     1029.0|        416.7| 3554.7|   4583.7|       15|              17|
|2026-02-08| 5554.0|      588.0|        555.4| 5521.4|   6109.4|       24|              21|
+----------+-------+-----------+-------------+-------+---------+---------+----------------+
only showing top 5 rows



# 5. GOLD: session_quality
**Columns:** date, avg_session_duration_secs, avg_pages_per_session,
bounce_rate, mobile_pct, desktop_pct, tablet_pct, top_countries (JSON)

In [0]:
# ⚡ Add date directly into session join using min(event_timestamp) → avoid distinct()
session_dates = (events_with_date
    .filter(F.col("session_id").isNotNull())
    .groupBy("session_id")
    .agg(F.min("date").alias("date")))   # ← groupBy instead of distinct() — faster

sessions_with_date = (sessions_df
    .join(session_dates, "session_id", "left")
    .filter(F.col("date").isNotNull()))

countries_df = (sessions_with_date
    .filter(F.col("country").isNotNull())
    .groupBy("date", "country").agg(F.count("*").alias("cnt"))
    .groupBy("date")
    .agg(F.to_json(F.map_from_entries(
        F.collect_list(F.struct("country", "cnt")))).alias("top_countries")))

session_quality = sessions_with_date.groupBy("date").agg(
    # ✅ FIX: session_duration_secs is now real (not NULL) with backdated timestamps
    F.round(F.avg("session_duration_secs"), 2).alias("avg_session_duration_secs"),
    F.round(F.avg("page_count"), 2).alias("avg_pages_per_session"),
    F.round(
        F.sum(F.col("is_bounce").cast("int")) / F.count("*"), 4
    ).alias("bounce_rate"),
    F.round(F.sum(F.when(F.col("device") == "mobile",  1).otherwise(0)) / F.count("*") * 100, 2).alias("mobile_pct"),
    F.round(F.sum(F.when(F.col("device") == "desktop", 1).otherwise(0)) / F.count("*") * 100, 2).alias("desktop_pct"),
    F.round(F.sum(F.when(F.col("device") == "tablet",  1).otherwise(0)) / F.count("*") * 100, 2).alias("tablet_pct"),
) \
.join(countries_df, "date", "left") \
.fillna({"top_countries": "{}"}) \
.select("date", "avg_session_duration_secs", "avg_pages_per_session",
        "bounce_rate", "mobile_pct", "desktop_pct", "tablet_pct", "top_countries") \
.orderBy("date")

session_quality.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_SESSION)
print(f"✅ Session Quality written: {session_quality.count()} rows")
session_quality.show(5)

print("\n🏆 ALL 5 GOLD TABLES WRITTEN SUCCESSFULLY!")

✅ Session Quality written: 94 rows
+----------+-------------------------+---------------------+-----------+----------+-----------+----------+--------------------+
|      date|avg_session_duration_secs|avg_pages_per_session|bounce_rate|mobile_pct|desktop_pct|tablet_pct|       top_countries|
+----------+-------------------------+---------------------+-----------+----------+-----------+----------+--------------------+
|2026-02-04|                 14540.93|                 1.51|     0.6241|     29.31|      35.86|     19.66|{"US":44,"DE":43,...|
|2026-02-05|                 11856.18|                 1.38|     0.6907|     25.83|      34.83|     24.32|{"SG":56,"IN":54,...|
|2026-02-06|                 15144.48|                 1.45|     0.6287|     31.92|      36.16|     18.57|{"IN":46,"US":65,...|
|2026-02-07|                 11091.43|                 1.31|     0.7318|     29.09|       35.0|     15.91|{"SG":33,"DE":32,...|
|2026-02-08|                 14847.82|                 1.39|     0.71

In [0]:
# ⚡ Free up Spark memory after all writes done
events_with_date.unpersist()
sessions_df.unpersist()
print("✅ Cache released — Gold job complete!")

✅ Cache released — Gold job complete!


## GOLD EXPORT: Plain Parquet for Snowpipe

In [0]:
export_base = "s3://realtime-saas-analytics-pipeline/gold_export"

# Fixed paths — overwrite same folder every run (no timestamps)
dau.write.format("parquet").mode("overwrite") \
    .save(f"{export_base}/daily_active_users/")

mrr.write.format("parquet").mode("overwrite") \
    .save(f"{export_base}/mrr_metrics/")

funnel.write.format("parquet").mode("overwrite") \
    .save(f"{export_base}/funnel_metrics/")

feature.write.format("parquet").mode("overwrite") \
    .save(f"{export_base}/feature_adoption/")

session_quality.write.format("parquet").mode("overwrite") \
    .save(f"{export_base}/session_quality/")

print("✅ Gold Export complete — fixed paths, no timestamps")

✅ Gold Export complete — fixed paths, no timestamps
